# 01 — Data Exploration

This notebook validates the 2023 season ingest before any modeling work begins.
It focuses on race coverage, lap completeness, compound quality, and other sanity checks.


In [15]:
from pathlib import Path
import sys
import os
import numpy as np
def find_backend_dir(start: Path) -> Path:

    candidates = [start] + list(start.parents)
    for p in candidates:
        maybe = p if p.name == "backend" else p / "backend"
        if (maybe / "ingestion").is_dir() and (maybe / ".env.example").exists():
            return maybe
    raise RuntimeError(
        f"Could not locate 'backend/' above {start}. "
        "Run this notebook from within the f1-strategy-lab repo."
    )

NOTEBOOK = Path.cwd()
BACKEND = find_backend_dir(NOTEBOOK)
PROJECT_ROOT = BACKEND.parent

sys.path.insert(0, str(BACKEND))

from dotenv import load_dotenv

env_path = BACKEND / ".env"
load_dotenv(env_path)

import pandas as pd
from sqlalchemy import create_engine, text
DATABASE_URL = os.getenv("DATABASE_URL")

if DATABASE_URL is None:
    raise RuntimeError(
        f"DATABASE_URL not found. Looked for it in {env_path}. "
        f"Does that file exist and does it define DATABASE_URL=...?"
    )

engine = create_engine(DATABASE_URL)
print(f"BACKEND resolved to: {BACKEND}")
print(f"Loaded .env from:    {env_path}  (exists: {env_path.exists()})")
print("Engine created OK.")


BACKEND resolved to: c:\Users\ASUS\Desktop\6-months projects\week 6 - end of phase 1\f1-strategy-lab\backend
Loaded .env from:    c:\Users\ASUS\Desktop\6-months projects\week 6 - end of phase 1\f1-strategy-lab\backend\.env  (exists: True)
Engine created OK.


## 1. Race coverage — do we have all 22 rounds?

In [16]:
races = pd.read_sql(text("SELECT id, year, round, name, circuit, total_laps FROM races ORDER BY round"), engine)
print(f"Rows in races: {len(races)}")
missing_rounds = sorted(set(range(1, 23)) - set(races["round"]))
print(f"Missing rounds (expected empty): {missing_rounds}")
races

Rows in races: 22
Missing rounds (expected empty): []


,id,year,round,name,circuit,total_laps
0,1,2023,1,Bahrain Grand Prix,Sakhir,57
1,3,2023,2,Saudi Arabian Grand Prix,Jeddah,50
2,4,2023,3,Australian Grand Prix,Melbourne,58
3,5,2023,4,Azerbaijan Grand Prix,Baku,51
4,6,2023,5,Miami Grand Prix,Miami,57
5,28,2023,6,Monaco Grand Prix,Monaco,78
6,8,2023,7,Spanish Grand Prix,Barcelona,66
7,9,2023,8,Canadian Grand Prix,Montréal,70
8,10,2023,9,Austrian Grand Prix,Spielberg,71
9,11,2023,10,British Grand Prix,Silverstone,52


## 2. Lap coverage per race
Same check `explore.py` does after ingestion: lap rows should be roughly
`total_laps * num_drivers`. A big shortfall usually means a partial/failed ingest for that round.

In [17]:
lap_counts = pd.read_sql(text("""
    SELECT race_id, COUNT(*) AS lap_rows, COUNT(DISTINCT driver_id) AS drivers_seen
    FROM laps
    GROUP BY race_id
"""), engine)

coverage = races.merge(lap_counts, left_on="id", right_on="race_id", how="left")
coverage["expected_floor"] = coverage["total_laps"] * 18
coverage["ratio"] = coverage["lap_rows"] / coverage["expected_floor"]
coverage["flag_under_ingested"] = coverage["ratio"] < 1.0

coverage[["round", "name", "total_laps", "lap_rows", "drivers_seen", "ratio", "flag_under_ingested"]] \
    .sort_values("round")

,round,name,total_laps,lap_rows,drivers_seen,ratio,flag_under_ingested
0,1,Bahrain Grand Prix,57,1056,20,1.029240,False
1,2,Saudi Arabian Grand Prix,50,943,20,1.047778,False
2,3,Australian Grand Prix,58,1003,20,0.960728,True
3,4,Azerbaijan Grand Prix,51,962,20,1.047930,False
4,5,Miami Grand Prix,57,1138,20,1.109162,False
5,6,Monaco Grand Prix,78,1515,20,1.079060,False
6,7,Spanish Grand Prix,66,1312,20,1.104377,False
7,8,Canadian Grand Prix,70,1317,20,1.045238,False
8,9,Austrian Grand Prix,71,1354,20,1.059468,False
9,10,British Grand Prix,52,971,20,1.037393,False


In [4]:
flagged = coverage[coverage["flag_under_ingested"]]
if len(flagged):
    print("Under-ingested rounds — re-run ingest.py for these:")
    print(flagged[["round", "name", "lap_rows", "expected_floor"]])
else:
    print("All 22 races pass the lap-count floor check.")

Under-ingested rounds — re-run ingest.py for these:
    round                   name  lap_rows  expected_floor
2       3  Australian Grand Prix      1003            1044
10     11   Hungarian Grand Prix      1252            1260
14     15   Singapore Grand Prix      1088            1116
15     16    Japanese Grand Prix       880             954
16     17       Qatar Grand Prix      1006            1026
19     20   São Paulo Grand Prix      1108            1278


## 3. Compound data quality

Some laps can legitimately have `compound` set to `NULL` or `'UNKNOWN'` because of in-laps, out-laps, or timing gaps.
This section measures how often that happens and flags races that deserve a closer look.


In [18]:
print(races.columns.tolist())

['id', 'year', 'round', 'name', 'circuit', 'total_laps']


In [19]:
compound_quality = pd.read_sql(
    text("""
        SELECT race_id,
               COUNT(*) AS recorded_laps,
               SUM(
                   CASE
                       WHEN compound IS NULL OR compound = 'UNKNOWN'
                       THEN 1
                       ELSE 0
                   END
               ) AS unknown_laps
        FROM laps
        GROUP BY race_id
    """),
    engine
)

# Merge with race metadata
compound_quality = races.merge(
    compound_quality,
    left_on="id",
    right_on="race_id",
    how="left"
)

# Replace missing values (if a race has no lap records)
compound_quality["recorded_laps"] = compound_quality["recorded_laps"].fillna(0)
compound_quality["unknown_laps"] = compound_quality["unknown_laps"].fillna(0)

# Compute percentage of unknown compounds
compound_quality["unknown_pct"] = np.where(
    compound_quality["recorded_laps"] > 0,
    (compound_quality["unknown_laps"] / compound_quality["recorded_laps"] * 100).round(2),
    0
)

# Flag races exceeding the threshold
flagged_compound = compound_quality[compound_quality["unknown_pct"] > 5.0]

print(f"Races over the 5% unknown-compound threshold: {len(flagged_compound)}")

# Display results
display(
    compound_quality[
        [
            "year",
            "round",
            "name",
            "total_laps",      # Official race length
            "recorded_laps",   # Lap records in the database
            "unknown_laps",
            "unknown_pct",
        ]
    ].sort_values("unknown_pct", ascending=False)
)

Races over the 5% unknown-compound threshold: 0


,year,round,name,total_laps,recorded_laps,unknown_laps,unknown_pct
7,2023,8,Canadian Grand Prix,70,1317,35,2.66
0,2023,1,Bahrain Grand Prix,57,1056,0,0.00
2,2023,3,Australian Grand Prix,58,1003,0,0.00
3,2023,4,Azerbaijan Grand Prix,51,962,0,0.00
4,2023,5,Miami Grand Prix,57,1138,0,0.00
1,2023,2,Saudi Arabian Grand Prix,50,943,0,0.00
5,2023,6,Monaco Grand Prix,78,1515,0,0.00
6,2023,7,Spanish Grand Prix,66,1312,0,0.00
8,2023,9,Austrian Grand Prix,71,1354,0,0.00
9,2023,10,British Grand Prix,52,971,0,0.00


## 4. Track status codes
Remember: `track_status` is a concatenated-digit **string**, not an integer, and can
contain multiple codes on one lap (e.g. `"24"` = yellow then SC in the same lap).
Any "clean lap" filter must check `track_status = '1'`, never `= 1`.

In [20]:
pd.read_sql(text("""
    SELECT track_status, COUNT(*) AS n
    FROM laps
    GROUP BY track_status
    ORDER BY n DESC
"""), engine)

,track_status,n
0,1,22011
1,12,795
2,4,469
3,41,244
4,124,215
5,21,127
6,671,108
7,14,82
8,126,77
9,45,39


## 5. Temperature columns

In [21]:
temp_check = pd.read_sql(text("""
    SELECT race_id,
           COUNT(*) AS total,
           SUM(CASE WHEN ambient_temp_c IS NULL THEN 1 ELSE 0 END) AS null_ambient,
           SUM(CASE WHEN track_temp_c IS NULL THEN 1 ELSE 0 END) AS null_track,
           SUM(CASE WHEN rainfall THEN 1 ELSE 0 END) AS rain_laps
    FROM laps
    GROUP BY race_id
"""), engine)
temp_check = races.merge(temp_check, left_on="id", right_on="race_id")
temp_check[["round", "name", "total", "null_ambient", "null_track", "rain_laps"]].sort_values("round")

,round,name,total,null_ambient,null_track,rain_laps
0,1,Bahrain Grand Prix,1056,0,0,0
1,2,Saudi Arabian Grand Prix,943,0,0,0
2,3,Australian Grand Prix,1003,0,0,0
3,4,Azerbaijan Grand Prix,962,0,0,0
4,5,Miami Grand Prix,1138,0,0,0
5,6,Monaco Grand Prix,1515,0,0,293
6,7,Spanish Grand Prix,1312,0,0,0
7,8,Canadian Grand Prix,1317,0,0,0
8,9,Austrian Grand Prix,1354,0,0,0
9,10,British Grand Prix,971,0,0,0


## 6. Lap time distribution by compound
A first look at whether compounds behave as expected (SOFT fastest, HARD slowest, on clean laps).
Filtered to `track_status = '1'` so SC/VSC-compressed laps don't distort this.

In [22]:
clean_laps = pd.read_sql(text("""
    SELECT l.compound, l.lap_time_ms
    FROM laps l
    WHERE l.track_status = '1'
      AND l.compound IS NOT NULL AND l.compound != 'UNKNOWN'
      AND l.lap_time_ms IS NOT NULL
"""), engine)

clean_laps.groupby("compound")["lap_time_ms"].describe()

,count,mean,std,min,25%,50%,75%,max
compound,,,,,,,,
HARD,10841.0,89760.652800,10520.330624,68111.0,81872.00,89279.0,98088.00,141611.0
INTERMEDIATE,524.0,90787.467557,7782.048277,80548.0,84911.00,88330.5,96359.00,129963.0
MEDIUM,7470.0,89816.574431,11740.884062,68150.0,80393.50,88566.5,98319.50,148490.0
SOFT,3023.0,88466.158783,13704.869042,67012.0,76538.00,81451.0,98827.50,133650.0
WET,32.0,98582.812500,4689.577445,90260.0,96152.75,98848.0,100205.75,118066.0


## 7. Pit stops sanity check

In [23]:
pit_stops = pd.read_sql(text("SELECT race_id, stop_duration_ms FROM pit_stops"), engine)
print(f"Total pit stop rows: {len(pit_stops)}")
print(f"Stops per race (avg): {len(pit_stops) / races['id'].nunique():.1f}")
pit_stops["stop_duration_ms"].describe()

Total pit stop rows: 538
Stops per race (avg): 24.5


count      538.000000
mean     24872.529740
std       6066.689084
min      16245.000000
25%      21912.250000
50%      23687.500000
75%      25854.750000
max      88943.000000
Name: stop_duration_ms, dtype: float64

## Summary
Before moving to `02_tire_degradation_model.ipynb`, confirm:
- [ ] All 22 rounds present, none flagged under-ingested
- [ ] Unknown-compound rate under ~5% everywhere (or you've decided which races to exclude)
- [ ] `track_status` values match the expected code table (1/2/4/5/6/7 combinations)
- [ ] `ambient_temp_c` / `track_temp_c` are populated (not universally NULL)
- [ ] Lap time distributions look physically sane per compound


In [24]:
pd.read_sql(
    text("""
        SELECT COUNT(*) AS string_none
        FROM laps
        WHERE compound = 'None';
    """),
    engine,
)

,string_none
0,0


## FINAL  — DATA VALIDATION CHECKS AFTER CORRECTION

### 1. PIT STOP VALIDATION

In [25]:
pit_stats = pd.read_sql("""
SELECT
    COUNT(*) AS total,
    MIN(stop_duration_ms) AS min_ms,
    MAX(stop_duration_ms) AS max_ms,
    AVG(stop_duration_ms) AS avg_ms
FROM pit_stops
""", engine)

pit_stats

,total,min_ms,max_ms,avg_ms
0,538,16245,88943,24872.52974


## 5. Temperature columns

These columns are reviewed separately to make sure they are complete and ready for later analysis.


In [26]:
outliers = pd.read_sql("""
SELECT *
FROM pit_stops
WHERE stop_duration_ms > 180000
ORDER BY stop_duration_ms DESC
""", engine)

outliers.head(20)

,id,race_id,driver_id,lap_number,stop_duration_ms,new_compound,new_tire_age


In [ ]:
race_pits = pd.read_sql("""
SELECT race_id, COUNT(*) as n_pits,
       AVG(stop_duration_ms) as avg_pit
FROM pit_stops
GROUP BY race_id
ORDER BY race_id
""", engine)

race_pits

,race_id,n_pits,avg_pit
0,1,30,26969.600000
1,3,23,21759.913043
2,4,29,554697.931034
3,5,22,21803.818182
4,6,19,24006.368421
5,8,29,23303.275862
6,9,28,26853.750000
7,10,29,19446.620690
8,11,23,29642.739130
9,12,26,22907.576923


In [27]:
pd.read_sql("""
SELECT compound, COUNT(*)
FROM laps
GROUP BY compound
ORDER BY COUNT(*) DESC
""", engine)

,compound,count
0,HARD,11827
1,MEDIUM,8311
2,SOFT,3475
3,INTERMEDIATE,724
4,WET,48
5,None,35


In [14]:
pd.read_sql("""
SELECT
    COUNT(*) FILTER (WHERE compound = 'None') AS string_none,
    COUNT(*) FILTER (WHERE compound IS NULL) AS real_null
FROM laps
""", engine)

,string_none,real_null
0,0,35
